# Week 4 - RAG Fundamentals with TechStore Plus

This notebook demonstrates the Week 4 RAG challenge using the existing TechStore chatbot domain. It loads local documents, splits them into chunks, stores embeddings in ChromaDB, reloads the vector store, and asks grounded questions.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

from src.rag.document_loader import load_documents
from src.rag.text_splitter import split_documents
from src.rag.vector_store import create_or_load_vector_store
from src.rag.rag_chain import TechStoreRAGAssistant

load_dotenv()

KNOWLEDGE_BASE = Path("docs/knowledge_base")
PERSIST_DIR = Path("chroma_db/techstore_knowledge")
COLLECTION = "techstore_knowledge"

## 1. Load Documents

In [ ]:
documents = load_documents(KNOWLEDGE_BASE)
len(documents), [doc.metadata["title"] for doc in documents]

## 2. Split Into Chunks

In [ ]:
chunks = split_documents(documents)
print(f"Generated {len(chunks)} chunks")
chunks[0]

## 3. Generate Embeddings And Store In ChromaDB

In [ ]:
vector_store = create_or_load_vector_store(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION,
)
vector_store.add_documents(chunks)
print(f"Persisted {len(chunks)} chunks in {PERSIST_DIR}")

## 4. Reload And Retrieve

In [ ]:
reloaded_store = create_or_load_vector_store(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION,
)
retrieved = reloaded_store.similarity_search("What is the return window for laptops?", k=3)
[(doc.metadata.get("title"), doc.page_content[:180]) for doc in retrieved]

## 5. Ask Grounded Questions

In [ ]:
assistant = TechStoreRAGAssistant(retriever=reloaded_store)

questions = [
    "What is the return window for laptops?",
    "How does warranty coverage work for smartphones?",
    "What shipping options are available?",
    "What should I do if my order arrived damaged?",
]

for question in questions:
    print("=" * 80)
    print(question)
    print(assistant.answer(question))